# Hybrid Quantum Model for Atmosphere Prediction

This notebook implements a wider hybrid auxiliary+spectral encoder -> re-uploading VQC -> target-wise regression head model that:
- Loads ADC2023 auxiliary metadata and 52-bin spectra from the local dataset copy
- Encodes auxiliary and spectral inputs separately, then fuses them with gated conditioning into a wider classical latent
- Projects that latent into a centered quantum interface of up to 20 qubits with repeated data re-uploading
- Uses richer quantum readout features rather than only one local measurement per qubit
- Predicts atmosphere parameters with a shared classical trunk plus target-specific heads that mix classical and quantum features

**Data:** Training from `data/ariel-ml-dataset/TrainingData/`, test from `data/ariel-ml-dataset/TestData/`.



## 1. Environment setup

In [1]:
# Standard library
import pickle
import random
import time
from collections import defaultdict
from pathlib import Path

# Scientific computing
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Machine learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Quantum (Qiskit)
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterExpression, ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient

# Adjoint gradient support from qiskit-algorithms
try:
    from qiskit_algorithms.gradients import ReverseEstimatorGradient
    from qiskit_algorithms.gradients.utils import GradientCircuit
    import qiskit_algorithms.gradients.utils as qiskit_grad_utils
    import qiskit_algorithms.gradients.base.base_estimator_gradient as qiskit_base_estimator_gradient
    _adjoint_available = True
except ImportError:
    _adjoint_available = False


def enable_qiskit_adjoint_patch() -> None:
    """Patch qiskit-algorithms so adjoint gradients work with symbolic CRX/CRY params.

    In the current stack, some controlled rotation gates still expose symbolic parameters
    while reporting ``is_parameterized() == False``. That leaves original ``theta[i]``
    symbols inside the internal gradient circuit and causes ``KeyError`` during backward.
    We patch the helper used by ``BaseEstimatorGradient._preprocess`` so every symbolic
    gate parameter gets a unique gradient parameter, regardless of that boolean.
    """

    if not _adjoint_available:
        return

    def _patched_assign_unique_parameters(circuit: QuantumCircuit) -> GradientCircuit:
        gradient_circuit = circuit.copy_empty_like(f"{circuit.name}_gradient")
        parameter_map = defaultdict(list)
        gradient_parameter_map = {}
        num_gradient_parameters = 0

        for instruction in circuit.data:
            op = (
                instruction.operation.to_mutable()
                if hasattr(instruction.operation, "to_mutable")
                else instruction.operation.copy()
            )
            op_params = list(getattr(op, "params", []))
            has_symbolic_params = any(getattr(angle, "parameters", None) for angle in op_params)

            if has_symbolic_params:
                new_op_params = []
                for angle in op_params:
                    if getattr(angle, "parameters", None):
                        new_parameter = Parameter(f"__gtheta{num_gradient_parameters}")
                        new_op_params.append(new_parameter)
                        num_gradient_parameters += 1
                        for parameter in angle.parameters:
                            parameter_map[parameter].append((new_parameter, angle.gradient(parameter)))
                        gradient_parameter_map[new_parameter] = angle
                    else:
                        new_op_params.append(angle)
                op.params = new_op_params

            gradient_circuit.append(op, instruction.qubits, instruction.clbits)

        gradient_circuit.global_phase = circuit.global_phase
        if isinstance(gradient_circuit.global_phase, ParameterExpression):
            substitution_map = {}
            for parameter in gradient_circuit.global_phase.parameters:
                if parameter in parameter_map:
                    substitution_map[parameter] = parameter_map[parameter][0][0]
                else:
                    new_parameter = Parameter(f"__gtheta{num_gradient_parameters}")
                    substitution_map[parameter] = new_parameter
                    parameter_map[parameter].append((new_parameter, 1))
                    num_gradient_parameters += 1
            gradient_circuit.global_phase = gradient_circuit.global_phase.subs(substitution_map)

        return GradientCircuit(gradient_circuit, parameter_map, gradient_parameter_map)

    qiskit_grad_utils._assign_unique_parameters = _patched_assign_unique_parameters
    qiskit_base_estimator_gradient._assign_unique_parameters = _patched_assign_unique_parameters


USE_ADJOINT_GRADIENT = _adjoint_available
if USE_ADJOINT_GRADIENT:
    enable_qiskit_adjoint_patch()
    print("Using ReverseEstimatorGradient (adjoint) with Qiskit compatibility patch.")
else:
    print("ReverseEstimatorGradient unavailable; using ParamShiftEstimatorGradient for backward.")



/Users/jkw/Documents/uni/axion/hack4sages/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using ReverseEstimatorGradient (adjoint) with Qiskit compatibility patch.


In [2]:
def set_random_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)

## 2. Data loading and preprocessing

Load auxiliary tables, FM targets, and 52-bin spectra keyed by `planet_ID`, then standardize each modality and build train/test tensors.



In [3]:
# Paths relative to project root
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent

candidate_roots = [
    PROJECT_ROOT / "data" / "ariel-ml-dataset",
    PROJECT_ROOT / "FullDataset",
]

DATA_ROOT = None
for candidate_root in candidate_roots:
    if (candidate_root / "TrainingData" / "AuxillaryTable.csv").exists():
        DATA_ROOT = candidate_root
        break

if DATA_ROOT is None:
    DATA_ROOT = candidate_roots[0]

AUX_TRAIN = DATA_ROOT / "TrainingData" / "AuxillaryTable.csv"
AUX_TEST = DATA_ROOT / "TestData" / "AuxillaryTable.csv"
SPEC_TRAIN = DATA_ROOT / "TrainingData" / "SpectralData.hdf5"
SPEC_TEST = DATA_ROOT / "TestData" / "SpectralData.hdf5"
FM_TRAIN = DATA_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv"

# Feature columns from auxiliary table (exclude planet_ID)
AUX_FEATURE_COLS = [
    "star_distance", "star_mass_kg", "star_radius_m", "star_temperature",
    "planet_mass_kg", "planet_orbital_period", "planet_distance", "planet_surface_gravity",
]

# Target columns from FM Parameter Table
TARGET_COLS = [
    "planet_radius", "planet_temp", "log_H2O", "log_CO2", "log_CO", "log_CH4", "log_NH3",
]

SPECTRAL_FIELDS = [
    "instrument_wlgrid",
    "instrument_spectrum",
    "instrument_noise",
    "instrument_width",
]


class SpectralStandardizer:
    def __init__(
        self,
        spectrum_mean: np.ndarray,
        spectrum_scale: np.ndarray,
        noise_mean: float,
        noise_scale: float,
        wl_mean: float,
        wl_scale: float,
        width_mean: float,
        width_scale: float,
    ) -> None:
        self.spectrum_mean = spectrum_mean.astype(np.float32)
        self.spectrum_scale = spectrum_scale.astype(np.float32)
        self.noise_mean = float(noise_mean)
        self.noise_scale = float(noise_scale)
        self.wl_mean = float(wl_mean)
        self.wl_scale = float(wl_scale)
        self.width_mean = float(width_mean)
        self.width_scale = float(width_scale)

    @classmethod
    def fit(cls, raw_spectra: np.ndarray) -> "SpectralStandardizer":
        raw64 = raw_spectra.astype(np.float64, copy=False)
        spectrum = raw64[:, :, 1]
        noise = raw64[:, :, 2]
        wl = raw64[0, :, 0]
        width = raw64[0, :, 3]

        spectrum_mean = spectrum.mean(axis=0)
        spectrum_scale = spectrum.std(axis=0)
        spectrum_scale = np.where(spectrum_scale == 0.0, 1.0, spectrum_scale)

        noise_mean = float(noise.mean())
        noise_scale = float(noise.std()) or 1.0
        wl_mean = float(wl.mean())
        wl_scale = float(wl.std()) or 1.0
        width_mean = float(width.mean())
        width_scale = float(width.std()) or 1.0

        return cls(
            spectrum_mean=spectrum_mean,
            spectrum_scale=spectrum_scale,
            noise_mean=noise_mean,
            noise_scale=noise_scale,
            wl_mean=wl_mean,
            wl_scale=wl_scale,
            width_mean=width_mean,
            width_scale=width_scale,
        )

    def transform(self, raw_spectra: np.ndarray) -> np.ndarray:
        spectrum = (raw_spectra[:, :, 1] - self.spectrum_mean) / self.spectrum_scale
        noise = (raw_spectra[:, :, 2] - self.noise_mean) / self.noise_scale
        wl = (raw_spectra[:, :, 0] - self.wl_mean) / self.wl_scale
        width = (raw_spectra[:, :, 3] - self.width_mean) / self.width_scale
        return np.stack([spectrum, noise, wl, width], axis=1).astype(np.float32)


def load_fm_targets(path: Path, columns: list[str]) -> pd.DataFrame:
    fm = pd.read_csv(path)
    if fm.columns[0] != "planet_ID" and "planet_ID" not in fm.columns:
        fm = fm.rename(columns={fm.columns[0]: "planet_ID"})
    return fm[["planet_ID", *columns]].copy()


def load_spectra(hdf5_path: Path, planet_ids: np.ndarray) -> np.ndarray:
    spectra = np.empty((len(planet_ids), 52, 4), dtype=np.float32)
    canonical_wl = None
    canonical_width = None

    with h5py.File(hdf5_path, "r") as handle:
        for idx, planet_id in enumerate(planet_ids):
            group = handle[f"Planet_{planet_id}"]
            wlgrid = group[SPECTRAL_FIELDS[0]][:].astype(np.float32)
            spectrum = group[SPECTRAL_FIELDS[1]][:].astype(np.float32)
            noise = group[SPECTRAL_FIELDS[2]][:].astype(np.float32)
            width = group[SPECTRAL_FIELDS[3]][:].astype(np.float32)

            if canonical_wl is None:
                canonical_wl = wlgrid
                canonical_width = width
            else:
                if not np.allclose(wlgrid, canonical_wl, atol=1.0e-8):
                    raise ValueError(f"{hdf5_path} has non-constant instrument_wlgrid.")
                if not np.allclose(width, canonical_width, atol=1.0e-8):
                    raise ValueError(f"{hdf5_path} has non-constant instrument_width.")

            spectra[idx, :, 0] = wlgrid
            spectra[idx, :, 1] = spectrum
            spectra[idx, :, 2] = noise
            spectra[idx, :, 3] = width

    return spectra



In [4]:
# Load CSVs and align targets on planet_ID
aux_train_df = pd.read_csv(AUX_TRAIN)
aux_test_df = pd.read_csv(AUX_TEST)
fm_train_df = load_fm_targets(FM_TRAIN, TARGET_COLS)
train_df = aux_train_df.merge(fm_train_df, on="planet_ID", how="inner")

print(f"Project root: {PROJECT_ROOT}")
print(f"Training samples after join: {len(train_df)}")
print(f"Auxiliary feature columns: {AUX_FEATURE_COLS}")
print(f"Target columns: {TARGET_COLS}")

train_planet_ids = train_df["planet_ID"].to_numpy()
test_planet_ids = aux_test_df["planet_ID"].to_numpy()
train_raw_spectra = load_spectra(SPEC_TRAIN, train_planet_ids)
test_raw_spectra = load_spectra(SPEC_TEST, test_planet_ids)

print(f"Training spectra shape: {train_raw_spectra.shape}")
print(f"Test spectra shape: {test_raw_spectra.shape}")



Project root: /Users/jkw/Documents/uni/axion/hack4sages
Training samples after join: 41423
Auxiliary feature columns: ['star_distance', 'star_mass_kg', 'star_radius_m', 'star_temperature', 'planet_mass_kg', 'planet_orbital_period', 'planet_distance', 'planet_surface_gravity']
Target columns: ['planet_radius', 'planet_temp', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
Training spectra shape: (41423, 52, 4)
Test spectra shape: (685, 52, 4)


In [5]:
# Extract feature and target arrays for training
X_train_aux = train_df[AUX_FEATURE_COLS].values.astype(np.float32)
y_train = train_df[TARGET_COLS].values.astype(np.float32)
X_test_aux = aux_test_df[AUX_FEATURE_COLS].values.astype(np.float32)

# Scale auxiliary features and targets
scaler_X = StandardScaler()
X_train_aux_scaled = scaler_X.fit_transform(X_train_aux)
X_test_aux_scaled = scaler_X.transform(X_test_aux)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)

# Standardize spectra into 4 channels: spectrum, noise, wavelength grid, instrument width
spectral_scaler = SpectralStandardizer.fit(train_raw_spectra)
X_train_spectra_scaled = spectral_scaler.transform(train_raw_spectra)
X_test_spectra_scaled = spectral_scaler.transform(test_raw_spectra)

# Convert to PyTorch tensors
X_train_aux_tensor = torch.tensor(X_train_aux_scaled, dtype=torch.float32)
X_train_spectra_tensor = torch.tensor(X_train_spectra_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_aux_tensor = torch.tensor(X_test_aux_scaled, dtype=torch.float32)
X_test_spectra_tensor = torch.tensor(X_test_spectra_scaled, dtype=torch.float32)

# Shared architecture and runtime config.
MAX_QUBITS = 20
NUM_QUBITS = 16
REUPLOAD_BLOCKS = 2
ENTANGLEMENT_STYLE = "ring_skip"
OBSERVABLE_STRATEGY = "z_zz"
AUX_EMBED_DIM = 32
SPECTRAL_HIDDEN_DIM = 96
SPECTRAL_EMBED_DIM = 64
FUSION_HIDDEN_DIM = 128
FUSED_LATENT_DIM = 96
QUANTUM_INPUT_DIM = NUM_QUBITS * REUPLOAD_BLOCKS
HEAD_SHARED_DIM = 96
TARGET_HEAD_HIDDEN_DIM = 64
QUANTUM_IMPLEMENTATION = "custom"
BATCH_SIZE = 16 if NUM_QUBITS > 12 else 32
EVAL_BATCH_SIZE = max(16, BATCH_SIZE)

if not 1 <= NUM_QUBITS <= MAX_QUBITS:
    raise ValueError(f"NUM_QUBITS must be in [1, {MAX_QUBITS}], got {NUM_QUBITS}.")
if REUPLOAD_BLOCKS < 1:
    raise ValueError("REUPLOAD_BLOCKS must be >= 1.")
if QUANTUM_INPUT_DIM != NUM_QUBITS * REUPLOAD_BLOCKS:
    raise ValueError("QUANTUM_INPUT_DIM must equal NUM_QUBITS * REUPLOAD_BLOCKS.")

train_dataset = TensorDataset(X_train_aux_tensor, X_train_spectra_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

n_train = X_train_aux_tensor.shape[0]
n_aux_features = X_train_aux_tensor.shape[1]
n_spectral_channels = X_train_spectra_tensor.shape[1]
n_spectral_bins = X_train_spectra_tensor.shape[2]
n_targets = len(TARGET_COLS)
print(f"Training size: {n_train}, aux features: {n_aux_features}, spectral tensor: {tuple(X_train_spectra_tensor.shape[1:])}")
print(f"Test samples: aux={X_test_aux_tensor.shape[0]}, spectra={X_test_spectra_tensor.shape[0]}")
print(
    f"Architecture config -> qubits={NUM_QUBITS}, reupload_blocks={REUPLOAD_BLOCKS}, "
    f"quantum_input_dim={QUANTUM_INPUT_DIM}, fused_latent={FUSED_LATENT_DIM}, "
    f"observable_strategy={OBSERVABLE_STRATEGY}"
)
print("Training path: CPU-first StatevectorEstimator with configurable quantum wrapper")



Training size: 41423, aux features: 8, spectral tensor: (4, 52)
Test samples: aux=685, spectra=685
Architecture config -> qubits=16, reupload_blocks=2, quantum_input_dim=32, fused_latent=96, observable_strategy=z_zz
Training path: CPU-first StatevectorEstimator with configurable quantum wrapper


## 3. Dual encoder and hybrid interface

Encode auxiliary metadata with an MLP and spectra with a residual multiscale 1D backbone, then use gated fusion so auxiliary metadata can modulate the spectral representation. Project the fused latent into a centered quantum input tensor with `REUPLOAD_BLOCKS * NUM_QUBITS` angles, and preserve modality-specific features for the final head.



In [6]:
class AuxEncoder(nn.Module):
    """MLP encoder for auxiliary tabular inputs."""

    def __init__(self, in_dim: int, hidden_dim: int = 64, out_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ResidualSpectralBlock(nn.Module):
    """Residual multiscale Conv1d block for short spectra."""

    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1):
        super().__init__()
        padding = dilation * (kernel_size - 1) // 2
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=kernel_size, padding=padding, dilation=dilation),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size=kernel_size, padding=padding, dilation=dilation),
        )
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.block(x))


class SpectralEncoder(nn.Module):
    """Residual multiscale 1D encoder with adaptive pooling for 52-bin spectra."""

    def __init__(self, in_channels: int = 4, hidden_dim: int = 96, out_dim: int = 64):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=1),
            nn.GELU(),
        )
        self.blocks = nn.Sequential(
            ResidualSpectralBlock(hidden_dim, kernel_size=5, dilation=1),
            ResidualSpectralBlock(hidden_dim, kernel_size=3, dilation=2),
            ResidualSpectralBlock(hidden_dim, kernel_size=3, dilation=4),
        )
        self.pool = nn.AdaptiveAvgPool1d(4)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        spec = self.input_proj(x)
        spec = self.blocks(spec)
        spec = self.pool(spec)
        return self.proj(spec)


class FusionEncoder(nn.Module):
    """FiLM-style gated fusion of auxiliary and spectral embeddings."""

    def __init__(self, aux_dim: int = 32, spec_dim: int = 64, hidden_dim: int = 128, out_dim: int = 96):
        super().__init__()
        self.film = nn.Linear(aux_dim, 2 * spec_dim)
        self.gate = nn.Sequential(
            nn.Linear(aux_dim + spec_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, spec_dim),
            nn.Sigmoid(),
        )
        self.fuse = nn.Sequential(
            nn.Linear(aux_dim + spec_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
        )

    def forward(self, aux_feat: torch.Tensor, spectral_feat: torch.Tensor) -> torch.Tensor:
        scale, shift = self.film(aux_feat).chunk(2, dim=-1)
        conditioned_spec = spectral_feat * (1.0 + 0.1 * scale) + shift
        gate = self.gate(torch.cat([aux_feat, conditioned_spec], dim=-1))
        gated_spec = gate * conditioned_spec + (1.0 - gate) * spectral_feat
        fused = torch.cat([aux_feat, gated_spec], dim=-1)
        return self.fuse(fused)


class QuantumProjection(nn.Module):
    """Project the fused latent into centered re-uploading quantum angles."""

    def __init__(self, in_dim: int = 96, hidden_dim: int = 128, out_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.net(x)) * np.pi



## 4. Re-uploading VQC (estimator + adjoint gradient)

Use a repeated data re-uploading circuit with configurable entanglement style. The quantum interface stays configurable up to `MAX_QUBITS`, and the default measurement bank combines local Pauli-Z terms with nearest-neighbor `ZZ` correlations.

In [7]:
def apply_entanglement(qc: QuantumCircuit, num_qubits: int, style: str = "ring_skip") -> None:
    """Apply a fixed entanglement pattern between re-uploading blocks."""
    if style not in {"ring", "ring_skip"}:
        raise ValueError(f"Unsupported entanglement style: {style}")

    for i in range(num_qubits):
        qc.cz(i, (i + 1) % num_qubits)

    if style == "ring_skip" and num_qubits > 3:
        for i in range(num_qubits):
            qc.cz(i, (i + 2) % num_qubits)


def add_data_reupload_block(
    qc: QuantumCircuit,
    data_params: ParameterVector,
    block_idx: int,
    num_qubits: int,
) -> None:
    """Inject a centered data slice into the circuit using two rotation axes."""
    offset = block_idx * num_qubits
    for i in range(num_qubits):
        angle = data_params[offset + i]
        qc.ry(angle, i)
        qc.rz(0.5 * angle, i)


def add_variational_block(
    qc: QuantumCircuit,
    weight_params: ParameterVector,
    weight_idx: int,
    num_qubits: int,
    entanglement_style: str,
) -> int:
    """Apply one trainable block after each data re-upload step."""
    for i in range(num_qubits):
        qc.ry(weight_params[weight_idx], i)
        weight_idx += 1
    for i in range(num_qubits):
        qc.rz(weight_params[weight_idx], i)
        weight_idx += 1

    apply_entanglement(qc, num_qubits, style=entanglement_style)

    for i in range(num_qubits):
        qc.rx(weight_params[weight_idx], i)
        weight_idx += 1

    return weight_idx


def create_reupload_circuit(
    num_qubits: int,
    num_blocks: int,
    entanglement_style: str = "ring_skip",
) -> tuple[QuantumCircuit, list, list]:
    """Build a repeated data re-uploading circuit and return circuit plus parameters."""
    data_params = ParameterVector("x", num_qubits * num_blocks)
    weight_params = ParameterVector("θ", 3 * num_qubits * num_blocks)
    qc = QuantumCircuit(num_qubits)

    weight_idx = 0
    for block_idx in range(num_blocks):
        add_data_reupload_block(qc, data_params, block_idx, num_qubits)
        weight_idx = add_variational_block(
            qc,
            weight_params,
            weight_idx,
            num_qubits,
            entanglement_style=entanglement_style,
        )

    return qc, list(data_params), list(weight_params)

In [8]:
# Keep the original circuit for backward so adjoint sees the true x/theta ordering.
class _AdjointEstimatorQNN(EstimatorQNN):
    def _backward(self, input_data, weights):
        parameter_values, num_samples = self._preprocess_forward(input_data, weights)
        input_grad, weights_grad = None, None
        if np.prod(parameter_values.shape) > 0:
            num_observables = self.output_shape[0]
            num_circuits = num_samples * num_observables
            circuits = [self._org_circuit] * num_circuits
            observables = [op for op in self._observables for _ in range(num_samples)]
            param_values = np.tile(parameter_values, (num_observables, 1))
            job = None
            if self._input_gradients:
                job = self.gradient.run(circuits, observables, param_values)
            elif len(parameter_values[0]) > self._num_inputs:
                params = [self._org_circuit.parameters[self._num_inputs :]] * num_circuits
                job = self.gradient.run(circuits, observables, param_values, parameters=params)
            if job is not None:
                from qiskit_machine_learning.exceptions import QiskitMachineLearningError
                try:
                    results = job.result()
                except Exception as exc:
                    raise QiskitMachineLearningError(f"Estimator job failed. {exc}") from exc
                input_grad, weights_grad = self._backward_postprocess(num_samples, results)
        return input_grad, weights_grad


def pauli_string(num_qubits: int, ops: dict[int, str]) -> str:
    chars = ["I"] * num_qubits
    for qubit, op in ops.items():
        chars[num_qubits - 1 - qubit] = op
    return "".join(chars)


def build_observables(num_qubits: int, strategy: str = "z_zz") -> list[SparsePauliOp]:
    observables = [
        SparsePauliOp.from_list([(pauli_string(num_qubits, {i: "Z"}), 1.0)])
        for i in range(num_qubits)
    ]

    if strategy == "z":
        return observables
    if strategy != "z_zz":
        raise ValueError(f"Unsupported observable strategy: {strategy}")

    if num_qubits == 2:
        zz_pairs = [(0, 1)]
    else:
        zz_pairs = [(i, (i + 1) % num_qubits) for i in range(num_qubits)]

    observables.extend(
        SparsePauliOp.from_list([(pauli_string(num_qubits, {i: "Z", j: "Z"}), 1.0)])
        for i, j in zz_pairs
    )
    return observables


def make_estimator():
    estimator = StatevectorEstimator()
    info = {
        "backend_source": "qiskit.primitives.StatevectorEstimator",
        "supports_adjoint": bool(USE_ADJOINT_GRADIENT),
    }
    return estimator, info


def make_qnn(
    num_qubits: int,
    num_blocks: int,
    input_gradients: bool = True,
    entanglement_style: str = "ring_skip",
    observable_strategy: str = "z_zz",
):
    qc, input_params, weight_params = create_reupload_circuit(
        num_qubits=num_qubits,
        num_blocks=num_blocks,
        entanglement_style=entanglement_style,
    )
    observables = build_observables(num_qubits, strategy=observable_strategy)
    estimator, estimator_info = make_estimator()

    if USE_ADJOINT_GRADIENT:
        gradient = ReverseEstimatorGradient()
        qnn_cls = _AdjointEstimatorQNN
        gradient_name = "adjoint"
    else:
        gradient = ParamShiftEstimatorGradient(estimator)
        qnn_cls = EstimatorQNN
        gradient_name = "param_shift"

    qnn = qnn_cls(
        circuit=qc,
        observables=observables,
        input_params=input_params,
        weight_params=weight_params,
        estimator=estimator,
        gradient=gradient,
        input_gradients=input_gradients,
    )

    info = {
        **estimator_info,
        "gradient_name": gradient_name,
        "num_qubits": num_qubits,
        "num_reupload_blocks": num_blocks,
        "entanglement_style": entanglement_style,
        "observable_strategy": observable_strategy,
        "num_quantum_weights": len(weight_params),
        "num_quantum_inputs": len(input_params),
        "num_observables": len(observables),
        "input_gradients": input_gradients,
    }
    return qnn, info

In [9]:
class _CustomQNNFunction(torch.autograd.Function):
    """Explicit Torch<->Qiskit bridge with input gradients enabled for hybrid training."""

    @staticmethod
    def forward(ctx, input_data: torch.Tensor, weights: torch.Tensor, qnn):
        ctx.qnn = qnn
        ctx.save_for_backward(input_data, weights)
        inp_np = input_data.detach().contiguous().cpu().numpy()
        w_np = weights.detach().contiguous().cpu().numpy()
        output = qnn.forward(inp_np, w_np)
        return torch.as_tensor(output, dtype=input_data.dtype, device=input_data.device)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        input_data, weights = ctx.saved_tensors
        qnn = ctx.qnn
        grad_output = grad_output.detach().contiguous()
        inp_np = input_data.detach().contiguous().cpu().numpy()
        w_np = weights.detach().contiguous().cpu().numpy()
        input_grad_np, weights_grad_np = qnn.backward(inp_np, w_np)

        input_grad = None
        if input_grad_np is not None:
            g = torch.as_tensor(input_grad_np, dtype=grad_output.dtype, device=input_data.device)
            if g.ndim == 2:
                g = g.unsqueeze(0)
            input_grad = torch.einsum("bo,boi->bi", grad_output, g)

        weights_grad = None
        if weights_grad_np is not None:
            g = torch.as_tensor(weights_grad_np, dtype=grad_output.dtype, device=weights.device)
            if g.ndim == 2:
                g = g.unsqueeze(0)
            weights_grad = torch.einsum("bo,bow->w", grad_output.to(weights.device), g)

        return input_grad, weights_grad, None


class CustomQNNLayer(nn.Module):
    def __init__(self, qnn):
        super().__init__()
        self.qnn = qnn
        initial_weights = 0.1 * torch.randn(qnn.num_weights, dtype=torch.float32)
        self.weights = nn.Parameter(initial_weights)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return _CustomQNNFunction.apply(x, self.weights, self.qnn)


class QuantumBlock(nn.Module):
    """Wraps EstimatorQNN for a configurable `(batch, REUPLOAD_BLOCKS * NUM_QUBITS)` interface."""

    def __init__(self, qnn, implementation: str = "custom"):
        super().__init__()
        if implementation == "torch_connector":
            self.quantum_layer = TorchConnector(qnn)
        elif implementation == "custom":
            self.quantum_layer = CustomQNNLayer(qnn)
        else:
            raise ValueError(f"Unknown quantum implementation: {implementation}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.quantum_layer(x)

## 5. Target-wise head with classical/quantum mixing

The prediction head keeps a shared classical trunk, learns how much each target should trust classical versus quantum features, and adds per-target residual subheads so all seven outputs do not need to share exactly the same final mixing rule.



In [10]:
class AtmosphereHead(nn.Module):
    """Shared trunk plus per-target heads with learned classical/quantum mixing."""

    def __init__(
        self,
        aux_dim: int = 32,
        spectral_dim: int = 64,
        classical_dim: int = 96,
        quantum_dim: int = 32,
        shared_dim: int = 96,
        target_hidden_dim: int = 64,
        n_targets: int = 7,
    ):
        super().__init__()
        self.n_targets = n_targets
        trunk_in_dim = aux_dim + spectral_dim + classical_dim
        self.shared_trunk = nn.Sequential(
            nn.Linear(trunk_in_dim, shared_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(shared_dim, shared_dim),
            nn.GELU(),
        )
        self.classical_logits = nn.Linear(shared_dim, n_targets)
        self.quantum_logits = nn.Linear(quantum_dim, n_targets)
        self.classical_residual = nn.Linear(classical_dim, n_targets)
        self.mix_gate = nn.Sequential(
            nn.Linear(shared_dim + quantum_dim, shared_dim),
            nn.GELU(),
            nn.Linear(shared_dim, n_targets),
            nn.Sigmoid(),
        )
        self.target_heads = nn.ModuleList(
            nn.Sequential(
                nn.Linear(shared_dim + quantum_dim, target_hidden_dim),
                nn.GELU(),
                nn.Linear(target_hidden_dim, 1),
            )
            for _ in range(n_targets)
        )

    def forward(
        self,
        aux_feat: torch.Tensor,
        spectral_feat: torch.Tensor,
        classical_feat: torch.Tensor,
        q_feat: torch.Tensor,
    ) -> torch.Tensor:
        shared_input = torch.cat([aux_feat, spectral_feat, classical_feat], dim=-1)
        shared_feat = self.shared_trunk(shared_input)
        gate = self.mix_gate(torch.cat([shared_feat, q_feat], dim=-1))
        mixed_logits = (1.0 - gate) * self.classical_logits(shared_feat) + gate * self.quantum_logits(q_feat)
        target_input = torch.cat([shared_feat, q_feat], dim=-1)
        target_residuals = torch.cat([head(target_input) for head in self.target_heads], dim=-1)
        return mixed_logits + target_residuals + self.classical_residual(classical_feat)

## 6. End-to-end hybrid model



In [11]:
class HybridAtmosphereModel(nn.Module):
    """Dual encoders -> gated fusion -> centered quantum projection -> VQC -> target-wise head."""

    def __init__(
        self,
        aux_encoder,
        spectral_encoder,
        fusion_encoder,
        quantum_projection,
        quantum_block,
        head,
    ):
        super().__init__()
        self.aux_encoder = aux_encoder
        self.spectral_encoder = spectral_encoder
        self.fusion_encoder = fusion_encoder
        self.quantum_projection = quantum_projection
        self.quantum_block = quantum_block
        self.head = head

    def forward(self, x_aux: torch.Tensor, x_spectra: torch.Tensor) -> torch.Tensor:
        aux_feat = self.aux_encoder(x_aux)
        spectral_feat = self.spectral_encoder(x_spectra)
        fused_latent = self.fusion_encoder(aux_feat, spectral_feat)
        quantum_inputs = self.quantum_projection(fused_latent)
        q_feat = self.quantum_block(quantum_inputs)
        return self.head(aux_feat, spectral_feat, fused_latent, q_feat)



In [12]:
# Build model components and full model
qnn, qnn_info = make_qnn(
    num_qubits=NUM_QUBITS,
    num_blocks=REUPLOAD_BLOCKS,
    input_gradients=True,
    entanglement_style=ENTANGLEMENT_STYLE,
    observable_strategy=OBSERVABLE_STRATEGY,
)
quantum_output_dim = qnn_info["num_observables"]

aux_encoder = AuxEncoder(in_dim=n_aux_features, hidden_dim=64, out_dim=AUX_EMBED_DIM)
spectral_encoder = SpectralEncoder(
    in_channels=n_spectral_channels,
    hidden_dim=SPECTRAL_HIDDEN_DIM,
    out_dim=SPECTRAL_EMBED_DIM,
)
fusion_encoder = FusionEncoder(
    aux_dim=AUX_EMBED_DIM,
    spec_dim=SPECTRAL_EMBED_DIM,
    hidden_dim=FUSION_HIDDEN_DIM,
    out_dim=FUSED_LATENT_DIM,
)
quantum_projection = QuantumProjection(
    in_dim=FUSED_LATENT_DIM,
    hidden_dim=max(FUSED_LATENT_DIM, 2 * QUANTUM_INPUT_DIM),
    out_dim=QUANTUM_INPUT_DIM,
)
quantum_block = QuantumBlock(qnn, implementation=QUANTUM_IMPLEMENTATION)
head = AtmosphereHead(
    aux_dim=AUX_EMBED_DIM,
    spectral_dim=SPECTRAL_EMBED_DIM,
    classical_dim=FUSED_LATENT_DIM,
    quantum_dim=quantum_output_dim,
    shared_dim=HEAD_SHARED_DIM,
    target_hidden_dim=TARGET_HEAD_HIDDEN_DIM,
    n_targets=n_targets,
)
model = HybridAtmosphereModel(
    aux_encoder,
    spectral_encoder,
    fusion_encoder,
    quantum_projection,
    quantum_block,
    head,
)

# Count parameters and summarize the chosen run path.
n_params = sum(p.numel() for p in model.parameters())
print(
    f"Model summary -> qubits={qnn_info['num_qubits']}, reupload_blocks={qnn_info['num_reupload_blocks']}, "
    f"entanglement={qnn_info['entanglement_style']}, observables={qnn_info['num_observables']}, "
    f"observable_strategy={qnn_info['observable_strategy']}, quantum_inputs={qnn_info['num_quantum_inputs']}, "
    f"quantum_weights={qnn_info['num_quantum_weights']}, gradient={qnn_info['gradient_name']}, "
    f"wrapper={QUANTUM_IMPLEMENTATION}"
)
print(f"Total trainable parameters: {n_params}")



Model summary -> qubits=16, reupload_blocks=2, entanglement=ring_skip, observables=32, observable_strategy=z_zz, quantum_inputs=32, quantum_weights=96, gradient=adjoint, wrapper=custom
Total trainable parameters: 416739


## 7. Training and evaluation

Use MSE on scaled targets. Validation split comes from the training set, while the public test split is kept for inference-only predictions. Validation and test inference are batched so wider quantum circuits can still be evaluated without pushing the full split through the QNN in one shot.



In [13]:
# Validation split from training data (20% for validation)
VAL_MONITOR_SAMPLES = 1024

indices = np.arange(n_train)
idx_train, idx_val = train_test_split(indices, test_size=0.2, random_state=RANDOM_SEED)

X_aux_tr = X_train_aux_tensor[idx_train]
X_spec_tr = X_train_spectra_tensor[idx_train]
y_tr = y_train_tensor[idx_train]
X_aux_val = X_train_aux_tensor[idx_val]
X_spec_val = X_train_spectra_tensor[idx_val]
y_val = y_train_tensor[idx_val]

train_loader = DataLoader(
    TensorDataset(X_aux_tr, X_spec_tr, y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_aux_val, X_spec_val, y_val),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)

if VAL_MONITOR_SAMPLES is not None and VAL_MONITOR_SAMPLES < len(y_val):
    val_subset_gen = torch.Generator().manual_seed(RANDOM_SEED)
    val_monitor_idx = torch.randperm(len(y_val), generator=val_subset_gen)[:VAL_MONITOR_SAMPLES]
    X_aux_val_monitor = X_aux_val[val_monitor_idx]
    X_spec_val_monitor = X_spec_val[val_monitor_idx]
    y_val_monitor = y_val[val_monitor_idx]
else:
    X_aux_val_monitor = X_aux_val
    X_spec_val_monitor = X_spec_val
    y_val_monitor = y_val

val_monitor_loader = DataLoader(
    TensorDataset(X_aux_val_monitor, X_spec_val_monitor, y_val_monitor),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)

print(f"Train batches: {len(train_loader)}, validation samples: {len(y_val)}")
print(f"Validation monitor samples per epoch: {len(y_val_monitor)}")
print(f"Eval batch size: {EVAL_BATCH_SIZE}")



Train batches: 2072, validation samples: 8285
Validation monitor samples per epoch: 1024
Eval batch size: 16


In [14]:
def train_epoch(model, loader, optimizer, loss_fn, epoch_idx=None, total_epochs=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    epoch_start = time.perf_counter()

    if epoch_idx is not None and total_epochs is not None:
        progress_desc = f"Epoch {epoch_idx + 1}/{total_epochs}"
    else:
        progress_desc = "Training"

    progress_bar = tqdm(loader, total=len(loader), desc=progress_desc, leave=False)
    for X_aux_b, X_spec_b, y_b in progress_bar:
        optimizer.zero_grad(set_to_none=True)
        pred = model(X_aux_b, X_spec_b)
        loss = loss_fn(pred, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        progress_bar.set_postfix(batch=n_batches, avg_loss=f"{total_loss / n_batches:.4f}")

    epoch_seconds = time.perf_counter() - epoch_start
    return total_loss / max(n_batches, 1), epoch_seconds


def predict_in_batches(model, loader):
    model.eval()
    preds = []
    with torch.inference_mode():
        for batch in loader:
            X_aux_b, X_spec_b = batch[:2]
            preds.append(model(X_aux_b, X_spec_b))
    return torch.cat(preds, dim=0)


def evaluate(model, loader, loss_fn, scaler_y=None):
    model.eval()
    eval_start = time.perf_counter()
    pred_batches = []
    y_batches = []

    with torch.inference_mode():
        for X_aux_b, X_spec_b, y_b in loader:
            pred_batches.append(model(X_aux_b, X_spec_b))
            y_batches.append(y_b)

    pred = torch.cat(pred_batches, dim=0)
    y = torch.cat(y_batches, dim=0)
    loss = loss_fn(pred, y).item()
    pred_np = pred.detach().cpu().numpy()
    y_np = y.detach().cpu().numpy()

    rmse_per_target = np.sqrt(np.mean((pred_np - y_np) ** 2, axis=0))
    if scaler_y is not None:
        pred_orig = scaler_y.inverse_transform(pred_np)
        y_orig = scaler_y.inverse_transform(y_np)
        rmse_orig = np.sqrt(np.mean((pred_orig - y_orig) ** 2, axis=0))
    else:
        rmse_orig = rmse_per_target

    eval_seconds = time.perf_counter() - eval_start
    return loss, rmse_per_target, rmse_orig, eval_seconds



In [15]:
EPOCHS = 20
LR = 1e-3
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
val_losses = []
train_epoch_times = []
val_times = []

In [16]:
print(
    f"Training uses batch size {BATCH_SIZE} with {len(train_loader)} batches per epoch; "
    f"eval batch size = {EVAL_BATCH_SIZE}; spectral input shape per sample = ({n_spectral_channels}, {n_spectral_bins})"
)
print(
    f"Hybrid widths -> fused_latent={FUSED_LATENT_DIM}, quantum_input={QUANTUM_INPUT_DIM}, "
    f"quantum_output={quantum_output_dim}, head_shared={HEAD_SHARED_DIM}, target_head_hidden={TARGET_HEAD_HIDDEN_DIM}"
)



Training uses batch size 16 with 2072 batches per epoch; eval batch size = 16; spectral input shape per sample = (4, 52)
Hybrid widths -> fused_latent=96, quantum_input=32, quantum_output=32, head_shared=96, target_head_hidden=64


In [17]:
for epoch in range(EPOCHS):
    tl, train_seconds = train_epoch(model, train_loader, optimizer, loss_fn, epoch_idx=epoch, total_epochs=EPOCHS)
    vl, rmse_scaled, rmse_orig, val_seconds = evaluate(
        model,
        val_monitor_loader,
        loss_fn,
        scaler_y,
    )

    train_losses.append(tl)
    val_losses.append(vl)
    train_epoch_times.append(train_seconds)
    val_times.append(val_seconds)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | Train loss: {tl:.6f} | "
        f"Monitor val loss: {vl:.6f} | Monitor val RMSE (orig): {rmse_orig.mean():.4f} | "
        f"Train: {train_seconds:.2f}s | Val: {val_seconds:.2f}s"
    )

print(
    f"Average train epoch time: {np.mean(train_epoch_times):.2f}s | "
    f"Average monitor validation time: {np.mean(val_times):.2f}s"
)



KeyboardInterrupt: 

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses, label="Monitor validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training and monitor-validation loss")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(train_epoch_times, label="Train seconds per epoch")
plt.plot(val_times, label="Monitor validation seconds")
plt.xlabel("Epoch")
plt.ylabel("Seconds")
plt.legend()
plt.title("Training and validation timing")
plt.tight_layout()
plt.show()

In [ ]:
# Final full-validation metrics (per-target RMSE in original scale)
_, _, rmse_orig, final_val_seconds = evaluate(model, val_loader, loss_fn, scaler_y)
print(f"Full validation time: {final_val_seconds:.2f}s")
print("Validation RMSE per target (original scale):")
for col, r in zip(TARGET_COLS, rmse_orig):
    print(f"  {col}: {r:.4f}")
print(f"Mean RMSE: {rmse_orig.mean():.4f}")



In [ ]:
# Save model and scalers for reuse
torch.save(model.state_dict(), "hybrid_atmosphere_model.pt")
with open("scaler_X.pkl", "wb") as f:
    pickle.dump(scaler_X, f)
with open("scaler_y.pkl", "wb") as f:
    pickle.dump(scaler_y, f)
with open("spectral_scaler.pkl", "wb") as f:
    pickle.dump(spectral_scaler, f)
print("Saved model state and all scalers.")



In [ ]:
# Inference on test set (no ground truth); save predictions if needed
test_loader = DataLoader(
    TensorDataset(X_test_aux_tensor, X_test_spectra_tensor),
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
)
test_pred_scaled = predict_in_batches(model, test_loader)
test_pred_orig = scaler_y.inverse_transform(test_pred_scaled.cpu().numpy())
print(f"Test set predictions shape: {test_pred_orig.shape}")
# Optional: save to CSV with planet_IDs from aux_test_df
# test_out = pd.DataFrame(test_pred_orig, columns=TARGET_COLS)
# test_out.insert(0, "planet_ID", aux_test_df["planet_ID"].values)
# test_out.to_csv("test_predictions.csv", index=False)

